# Step 14 — Full projected-drift replacement, and CNN-labelled bias for every checkpoint

**Why.** Step 12F showed that subtracting only the asymmetry coefficient `h_net − h_exact` removes part of the class bias, and that LDA-sign labels fail for undertrained models. Two things follow.

**A. Decisive intervention.** Replace the network's *whole* projected drift along `u` with the exact one: `D'(x) = D(x) + σ²·[g_exact(m;σ) − g_net(m;σ)]/ℓ · u`, where `g` is the binned 1-D drift (score units of m). Everything orthogonal to `u` is untouched. If this removes the bias for all samplers, the bias lives in the 1-D projection.

**B. CNN-labelled bias for all 26 checkpoints**, Heun-40 and stochastic-40, n=4000 per cell, with the real-data CNN baseline. Replaces the LDA-labelled numbers of Steps 8–10.

Mount the Step 7 and Step 10 notebook outputs. GPU on, ~25 min.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, glob, os
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev); rng = np.random.default_rng(0); SIGMA_DATA = 0.5
def load(ds_cls, root):
    d = ds_cls(root, train=True, download=True); return (d.data.numpy().astype(np.float32)/127.5 - 1.0).reshape(-1, 784), d.targets.numpy()
X_MN, Y_MN = load(datasets.MNIST, '/kaggle/working/mnist'); X_FM, Y_FM = load(datasets.FashionMNIST, '/kaggle/working/fmnist')
GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
def smooth_hist(m0, tau, grid=GRID):
    dg = grid[1]-grid[0]; h, _ = np.histogram(m0, bins=np.append(grid-dg/2, grid[-1]+dg/2), density=True); k = np.exp(-grid**2/(2*tau)); k /= k.sum()
    return np.maximum(fftconvolve(h, k, mode='same'), 1e-300)
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def balanced(X, Y, pair):
    Xa = X[Y == pair[0]]; Xb = X[Y == pair[1]]; n = min(len(Xa), len(Xb)); return Xa[:n], Xb[:n]
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__(); self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1); self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]; return x + self.c2(F.silu(self.n2(h)))
class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1); self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1); self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31)); e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e); return self.out(self.u1(self.up(h2) + h1, e))
def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1); c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))
def load_net(path): net = UNet().to(dev); net.load_state_dict(torch.load(path, map_location=dev)); return net.eval()
class CNN(nn.Module):
    def __init__(self):
        super().__init__(); self.c1 = nn.Conv2d(1, 32, 3, padding=1); self.c2 = nn.Conv2d(32, 64, 3, padding=1); self.fc = nn.Linear(64*7*7, 2)
    def forward(self, x): x = F.max_pool2d(F.relu(self.c1(x)), 2); x = F.max_pool2d(F.relu(self.c2(x)), 2); return self.fc(x.flatten(1))
def train_cnn(Xa, Xb, epochs=3):
    X = torch.from_numpy(np.concatenate([Xa, Xb])).view(-1, 1, 28, 28).to(dev); Y = torch.cat([torch.zeros(len(Xa)), torch.ones(len(Xb))]).long().to(dev)
    net = CNN().to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(X), device=dev)
        for i in range(0, len(X), 256): idx = perm[i:i+256]; loss = F.cross_entropy(net(X[idx]), Y[idx]); opt.zero_grad(); loss.backward(); opt.step()
    net.eval(); return net

S_MAX, S_MIN, RHO = 20.0, 0.002, 7.0
SIG_TAB = np.geomspace(0.05, S_MAX, 48)          # sigma table for the drift-replacement lookup
@torch.no_grad()
def drift_tables(net, Xflat, u, mid, half, n=20000, nbins=80):
    """For each sigma in SIG_TAB: binned network drift g_net(m) and exact drift g_exact(m) on a common grid."""
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); m0 = (Xflat-mid) @ u / half; tabs = []
    for sigma in SIG_TAB:
        x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
        D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), float(sigma), device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
        m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
        lo, hi = np.quantile(m, [0.001, 0.999]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
        cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
        cen = 0.5*(edges[:-1]+edges[1:]); gnet = np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9)
        c = np.linspace(min(lo, -30), max(hi, 30), 12001); p = smooth_hist(m0, sigma**2/half**2, grid=c); gex = np.interp(cen, c, np.gradient(np.log(p), c))
        tabs.append((cen, gex - gnet))                                # correction = exact - network, in score units of m
    return tabs
def make_corr(tabs):
    def corr(m_np, sigma):
        k = int(np.argmin(np.abs(np.log(SIG_TAB) - np.log(sigma)))); cen, dg = tabs[k]; return np.interp(m_np, cen, dg)
    return corr
@torch.no_grad()
def sample(net, u, mid, half, corr=None, steps=40, churn=0.0, n=4000):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); ut4 = ut.view(1, 1, 28, 28)
    def Dc(x, s):
        D = denoise(net, x, s.expand(n))
        if corr is None: return D
        m = ((x.view(n, -1) - midt) @ ut / half).cpu().numpy(); dg = torch.from_numpy(corr(m, float(s)).astype(np.float32)).to(dev)
        return D + (float(s)**2/half)*dg.view(n, 1, 1, 1)*ut4       # D' = D + sigma^2 (g_exact - g_net)/half * u
    i = torch.arange(steps, device=dev); sig = torch.cat([(S_MAX**(1/RHO) + i/(steps-1)*(S_MIN**(1/RHO)-S_MAX**(1/RHO)))**RHO, torch.zeros(1, device=dev)])
    x = torch.randn(n, 1, 28, 28, device=dev)*sig[0]
    for k in range(steps):
        s0, s1 = sig[k], sig[k+1]
        if churn > 0 and 0.01 <= s0 <= S_MAX:
            gam = min(churn/steps, 2**0.5-1); sh = s0*(1+gam); x = x + torch.sqrt(sh**2 - s0**2)*torch.randn_like(x); s0 = sh
        d0 = (x - Dc(x, s0))/s0; x1 = x + (s1-s0)*d0
        if s1 > 0: d1 = (x1 - Dc(x1, s1))/s1; x = x + (s1-s0)*0.5*(d0+d1)
        else: x = x1
    return x
CK = sorted(glob.glob('/kaggle/input/**/unet_*.pt', recursive=True)); print(len(CK), 'checkpoints')

## A. Replace the whole projected drift along u with the exact one

In [ ]:
def pick(pair, ep, fm=False):
    c = [p for p in CK if (f'unet_{"fm" if fm else ""}{pair[0]}{pair[1]}' in os.path.basename(p)) and f'_ep{ep}.pt' in p and ('_s' not in os.path.basename(p) or '_s0_' in p)]
    return c[0]
print(f"{'pair':>6} {'ep':>4} {'sampler':>8} {'frac+ CNN, network':>19} {'frac+ CNN, drift replaced':>26} {'baseline':>9}")
for pair in [(0, 1), (0, 8), (3, 8)]:
    Xa, Xb = balanced(X_MN, Y_MN, pair); u, mid, half = lda_direction(Xa, Xb); Xflat = np.concatenate([Xa, Xb]); cnn = train_cnn(Xa, Xb)
    base = float((cnn(torch.from_numpy(Xflat).view(-1,1,28,28).to(dev)).argmax(1) == 0).float().mean())
    for ep in [20, 80]:
        net = load_net(pick(pair, ep)); corr = make_corr(drift_tables(net, Xflat, u, mid, half))
        for sname, kw in [('heun40', dict(steps=40)), ('sde10', dict(steps=10, churn=40)), ('sde40', dict(steps=40, churn=40)), ('sde80', dict(steps=80, churn=40))]:
            f0 = float((cnn(sample(net, u, mid, half, None, **kw)).argmax(1) == 0).float().mean()); f1 = float((cnn(sample(net, u, mid, half, corr, **kw)).argmax(1) == 0).float().mean())
            print(f"{pair!s:>6} {ep:4d} {sname:>8} {f0:19.3f} {f1:26.3f} {base:9.3f}")

## B. CNN-labelled class fraction for every checkpoint (replaces LDA-labelled numbers)

In [ ]:
print(f"{'checkpoint':>26} {'heun40 CNN':>11} {'sde40 CNN':>10} {'heun40 LDA':>11} {'baseline CNN':>13}")
cache = {}
for path in CK:
    name = os.path.basename(path).replace('.pt', ''); tag = name.replace('unet_', ''); fm = tag.startswith('fm'); tag = tag[2:] if fm else tag
    pair = (int(tag[0]), int(tag[1])); X, Y = (X_FM, Y_FM) if fm else (X_MN, Y_MN)
    key = (pair, fm)
    if key not in cache:
        Xa, Xb = balanced(X, Y, pair); u, mid, half = lda_direction(Xa, Xb); cnn = train_cnn(Xa, Xb)
        base = float((cnn(torch.from_numpy(np.concatenate([Xa, Xb])).view(-1,1,28,28).to(dev)).argmax(1) == 0).float().mean()); cache[key] = (u, mid, half, cnn, base)
    u, mid, half, cnn, base = cache[key]; net = load_net(path); ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev)
    xh = sample(net, u, mid, half, None, steps=40); xs = sample(net, u, mid, half, None, steps=40, churn=40)
    fh = float((cnn(xh).argmax(1) == 0).float().mean()); fs = float((cnn(xs).argmax(1) == 0).float().mean()); fl = float((((xh.view(-1, 784)-midt) @ ut / half) > 0).float().mean())
    print(f"{name:>26} {fh:11.3f} {fs:10.3f} {fl:11.3f} {base:13.3f}")

## What to report back
**A.** For each pair/epoch/sampler: network fraction vs drift-replaced fraction (CNN labels). If replacement brings every row to the baseline, the bias lives in the projected drift. Partial removal quantifies how much does.
**B.** The per-checkpoint table. This is the source for the revised bias figure and the seed table.

Note: n=4000 per cell, so the binomial standard error is about 0.008.